In [1]:
#import packages
from pprint import pprint
import tensorflow as tf
import tensorflow_probability as tfp
import keras
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts
import aneurysm_3Dsegmentation_in_CTA.model_utility.configuration as conf
import aneurysm_3Dsegmentation_in_CTA.model_utility.metrics as metrics
import aneurysm_3Dsegmentation_in_CTA.model_utility.tuning as tuner
import segmentation_models_3D as sm3
import keras_tuner as kt
import os

I0000 00:00:1793945958.119530   17961 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Segmentation Models: using `tf.keras` framework.


In [2]:
# loading data tensors
dataSource = "data/CTA nii" 

sizeContainer = tts.dataSplitPerSize(dataSource)

_ , testSet , _ = tts.dataSplitPerSample(
    [
        sizeContainer["S"] ,
        sizeContainer["M"] ,
        sizeContainer["L"]
    ] ,
    testRatio=0.2 ,
    seed=42 ,
    perSet=True
)

imgTestOver  , labelTestOver  , mergeTestOver  = tts.dataTensorLoading(testSet)


perSize = tts.dataTensorloadingPerSize(testSet)
imgSmallSet , labelSmallSet , mergeSmallSet = perSize["S"]
imgMedSet   , labelMedSet   , mergeMedSet   = perSize["M"]
imgLargeSet , labelLargeSet , mergeLargeSet = perSize["L"]

In [ ]:
print("//////////////////////////////////////////////// Small ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgSmallSet , labelSmallSet , mergeSmallSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")
print("//////////////////////////////////////////////// Medium ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgMedSet , labelMedSet , mergeMedSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")
print("//////////////////////////////////////////////// Large ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgLargeSet , labelLargeSet , mergeLargeSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")

In [ ]:
# pipeline configuring

geo      = utl.randomGeo(p=0.5)
crop     = utl.volume_crop((128 , 128 , 128) , paddDim=80) 
tile     = utl.tile(
    tile_dim=[1 , 1 , 1 , 1 , 3]
)

setShape = utl.setShape(
    imgShape=[None , 128 , 128 , 128 , 3] ,   
    labelShape=[None , 128 , 128 , 128 , 1]   
)


# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64 , tf.float64])
def rimg(imgPath , labelPath , mergePath) :
    return utl.read_img(imgPath , labelPath , mergePath)
def read_img(img , label , merge) :
    imglbl = rimg(img , label , merge)
    img   = imglbl[0]
    label = imglbl[1]
    merge = imglbl[2]
    return img , label , merge

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label


# windowing configuration part
normalizer    = utl.fixedRangeNormalization(min_val=0 , max_val=3071 , clip=True)
fixwindower   = window.windowing(ww=395 , wl=274)

In [ ]:
dataloaderValid = (
    tf.data.Dataset.from_tensor_slices((imgLargeSet , labelLargeSet , mergeLargeSet)) 

    .map(
        read_img , 
        num_parallel_calls=2
    )
    .map(
        crop.cropping ,
        num_parallel_calls=2
    )
    .map(  # here is different for quartile windowing (cheannlize)
        utl.channelize ,
        num_parallel_calls=2
    )
    .cache() 
    .batch(batch_size=2)
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        fixwindower.apply ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

In [ ]:
# test
weights = ""

# loss define
binaryFocalLoss = sm3.losses.binary_focal_loss
diceLoss        = sm3.losses.dice_loss 
weightedBinaryFocalDiceLoss = conf.WeightedSumOfLosses(binaryFocalLoss , diceLoss , alpha=[1.3 , 0.2])

# model
model = sm3.Unet(
    backbone_name="seresnet18" , 
    input_shape=(128 , 128 , 128 , 3) , # change
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features="default"
)

# loading weights
model.load_weights(
    weights
)

# compiling
model.compile(
    loss = weightedBinaryFocalDiceLoss ,
    metrics=[
        metrics.V_Recall ,
        metrics.dice ,
        metrics.precision ,
    ] ,
)

In [ ]:
eval = model.evaluate(
    dataloaderValid
)

In [ ]:
recall = []
dice   = []
prec   = []
f1     = []  

for data in dataloaderValid :
    images = data[0]
    labels = data[1]
    #-----
    pred = model.predict(
        images
    )

    # evaluation
    Recall = metrics.V_Recall(
        labels ,
        pred
    )
    recall.append(tf.reduce_mean(Recall))

    Dice = metrics.dice(
        labels ,
        pred
    )
    dice.append(tf.reduce_mean(Dice))

    Precision = metrics.precision(
        labels ,
        pred
    )
    prec.append(tf.reduce_mean(Precision))

print("recall : " , np.mean(recall) , " std : " , np.std(recall))
print("dice : "   , np.mean(dice)   , " std : " , np.std(dice))
print("prec : "   , np.mean(prec)   , " std : " , np.std(prec))